# 08 Profile-Guided Feature Importance and Pruning

Goal: read the pandas profiling JSON, identify suspicious and useful features, retrain the latest tuned LightGBM setup, inspect feature importance carefully, prune weak features, and evaluate the final feature list. This notebook does not modify previous notebooks or the previous LightGBM setup.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from src.features import BASIC_APPLICATION_FEATURES, create_features
from src.preprocessing import build_preprocessor
from src.thresholding import (
    evaluate_probabilities,
    find_best_threshold,
    get_positive_probabilities,
)


In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
VALIDATION_SIZE = 0.2

LIGHTGBM_PARAMS = {
    "subsample": 1.0,
    "reg_lambda": 0.0,
    "num_leaves": 31,
    "n_estimators": 600,
    "min_child_samples": 50,
    "learning_rate": 0.02,
    "colsample_bytree": 0.85,
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1,
    "importance_type": "gain",
}

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "reports"
PROFILE_JSON_PATH = REPORTS_DIR / "application_train_profile.json"

train_df = pd.read_csv(
    RAW_DATA_DIR / "application_train.csv"
)

y = train_df["TARGET"].copy()

with PROFILE_JSON_PATH.open(encoding="utf-8") as file:
    profile_json = json.load(file)

len(profile_json["variables"]), train_df.shape


(122, (307511, 122))

## Read the Profiling JSON

The profiling JSON is used as a data-quality scan. It covers the raw `application_train.csv` columns, so engineered features are evaluated later through model importance rather than profiling metadata.


In [3]:
profile_rows = []

for feature, stats in profile_json["variables"].items():
    value_counts = stats.get("value_counts_without_nan") or {}
    n = stats.get("n", 0) or 0
    n_missing = stats.get("n_missing", 0) or 0
    max_count = max(value_counts.values()) if value_counts else 0
    dominant_ratio = max(max_count, n_missing) / n if n else np.nan

    profile_rows.append({
        "feature": feature,
        "type": stats.get("type"),
        "n": n,
        "n_missing": n_missing,
        "p_missing": stats.get("p_missing", 0),
        "n_distinct": stats.get("n_distinct"),
        "p_distinct": stats.get("p_distinct"),
        "is_unique": stats.get("is_unique"),
        "dominant_ratio": dominant_ratio,
        "min": stats.get("min"),
        "max": stats.get("max"),
        "mean": stats.get("mean"),
        "std": stats.get("std"),
        "skewness": stats.get("skewness"),
        "p_zeros": stats.get("p_zeros", 0),
    })

profile_df = pd.DataFrame(profile_rows)
profile_df.head()


,feature,type,n,n_missing,p_missing,n_distinct,p_distinct,is_unique,dominant_ratio,min,max,mean,std,skewness,p_zeros
0,SK_ID_CURR,Numeric,307511,0,0.0,307511,1.000000,True,0.000003,100002.0,456255.0,278180.518577,102790.175348,-0.001200,0.000000
1,TARGET,Numeric,307511,0,0.0,2,0.000007,False,0.919271,0.0,1.0,0.080729,0.272419,3.078159,0.919271
2,NAME_CONTRACT_TYPE,Text,307511,0,0.0,2,0.000007,False,0.904787,NaN,NaN,NaN,NaN,NaN,0.000000
3,CODE_GENDER,Text,307511,0,0.0,3,0.000010,False,0.658344,NaN,NaN,NaN,NaN,NaN,0.000000
4,FLAG_OWN_CAR,Text,307511,0,0.0,2,0.000007,False,0.659892,NaN,NaN,NaN,NaN,NaN,0.000000


In [4]:
profile_issue_rows = []

for _, row in profile_df.iterrows():
    issues = []

    if row["feature"] in ["SK_ID_CURR", "TARGET"]:
        issues.append("identifier_or_target")

    if row["p_missing"] >= 0.50:
        issues.append("high_missing")
    elif row["p_missing"] >= 0.25:
        issues.append("moderate_missing")

    if row["dominant_ratio"] >= 0.995:
        issues.append("near_constant")

    if bool(row["is_unique"]):
        issues.append("unique_identifier_like")

    if abs(row["skewness"] or 0) >= 10:
        issues.append("extreme_skew")

    if row["feature"] == "DAYS_EMPLOYED" and row["max"] == 365243:
        issues.append("known_sentinel_365243")

    if issues:
        profile_issue_rows.append({
            **row.to_dict(),
            "issues": ", ".join(issues),
            "issue_count": len(issues),
        })

profile_issues_df = pd.DataFrame(profile_issue_rows).sort_values(
    ["issue_count", "p_missing", "dominant_ratio"],
    ascending=False,
).reset_index(drop=True)

profile_issues_df[[
    "feature",
    "type",
    "p_missing",
    "dominant_ratio",
    "skewness",
    "issues",
]].head(30)


,feature,type,p_missing,dominant_ratio,skewness,issues
0,NONLIVINGAPARTMENTS_AVG,Numeric,0.694330,0.694330,15.541185,"high_missing, extreme_skew"
1,NONLIVINGAPARTMENTS_MODE,Numeric,0.694330,0.694330,16.251819,"high_missing, extreme_skew"
2,NONLIVINGAPARTMENTS_MEDI,Numeric,0.694330,0.694330,15.671995,"high_missing, extreme_skew"
3,YEARS_BEGINEXPLUATATION_AVG,Numeric,0.487810,0.487810,-15.515264,"moderate_missing, extreme_skew"
4,YEARS_BEGINEXPLUATATION_MODE,Numeric,0.487810,0.487810,-14.755318,"moderate_missing, extreme_skew"
5,YEARS_BEGINEXPLUATATION_MEDI,Numeric,0.487810,0.487810,-15.573124,"moderate_missing, extreme_skew"
6,FLAG_MOBIL,Numeric,0.000000,0.999997,-554.536744,"near_constant, extreme_skew"
7,FLAG_DOCUMENT_12,Numeric,0.000000,0.999993,392.114779,"near_constant, extreme_skew"
8,FLAG_DOCUMENT_10,Numeric,0.000000,0.999977,209.589054,"near_constant, extreme_skew"
9,FLAG_DOCUMENT_2,Numeric,0.000000,0.999958,153.791817,"near_constant, extreme_skew"


In [5]:
profile_useful_candidates = [
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "AMT_INCOME_TOTAL",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "CNT_FAM_MEMBERS",
    "CNT_CHILDREN",
]

profile_df.loc[
    profile_df["feature"].isin(profile_useful_candidates),
    ["feature", "type", "p_missing", "dominant_ratio", "skewness", "min", "max"],
].sort_values("feature")


,feature,type,p_missing,dominant_ratio,skewness,min,max
9,AMT_ANNUITY,Numeric,0.000039,0.020763,1.579777,1.615500e+03,2.580255e+05
8,AMT_CREDIT,Numeric,0.000000,0.031573,1.234778,4.500000e+04,4.050000e+06
10,AMT_GOODS_PRICE,Numeric,0.000904,0.084621,1.349000,4.050000e+04,4.050000e+06
7,AMT_INCOME_TOTAL,Numeric,0.000000,0.116256,391.559654,2.565000e+04,1.170000e+08
6,CNT_CHILDREN,Numeric,0.000000,0.700368,1.974604,0.000000e+00,1.900000e+01
29,CNT_FAM_MEMBERS,Numeric,0.000007,0.514964,0.987543,1.000000e+00,2.000000e+01
17,DAYS_BIRTH,Numeric,0.000000,0.000140,-0.115673,-2.522900e+04,-7.489000e+03
18,DAYS_EMPLOYED,Numeric,0.000000,0.180072,1.664346,-1.791200e+04,3.652430e+05
41,EXT_SOURCE_1,Numeric,0.563811,0.563811,-0.068755,1.456813e-02,9.626928e-01
42,EXT_SOURCE_2,Numeric,0.002146,0.002345,-0.793576,8.173617e-08,8.549997e-01


## Interpretation Before Modeling

Most buggy-looking raw features are building/property columns with very high missingness, near-constant document/contact flags, the unique ID, and the `DAYS_EMPLOYED` sentinel value. The current curated feature set already avoids most high-missing building columns and handles the employment sentinel in `create_features()`. Likely useful raw features are the `EXT_SOURCE_*` variables, core credit/income amount variables, age/employment variables, and family-size variables. Engineered ratios from these variables must be tested by model importance.


In [6]:
X_raw = train_df[BASIC_APPLICATION_FEATURES].copy()
X = create_features(X_raw)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALIDATION_SIZE,
    stratify=y_train_full,
    random_state=RANDOM_STATE,
)

numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

X.shape, len(numeric_columns), len(categorical_columns)


((307511, 32), 26, 6)

In [7]:
def build_lgbm_pipeline(feature_columns):
    local_numeric_columns = [
        column for column in feature_columns
        if column in numeric_columns
    ]
    local_categorical_columns = [
        column for column in feature_columns
        if column in categorical_columns
    ]

    preprocessor = build_preprocessor(
        numeric_columns=local_numeric_columns,
        categorical_columns=local_categorical_columns,
        one_hot_sparse_output=True,
    )

    model = LGBMClassifier(**LIGHTGBM_PARAMS)

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )


def evaluate_feature_list(feature_columns, label):
    pipeline = build_lgbm_pipeline(feature_columns)
    pipeline.fit(X_train[feature_columns], y_train)

    valid_probabilities = get_positive_probabilities(
        pipeline,
        X_valid[feature_columns],
    )

    threshold_info = find_best_threshold(
        y_true=y_valid,
        probabilities=valid_probabilities,
    )

    valid_metrics = evaluate_probabilities(
        y_true=y_valid,
        probabilities=valid_probabilities,
        threshold=threshold_info["threshold"],
    )

    return {
        "label": label,
        "pipeline": pipeline,
        "threshold": threshold_info["threshold"],
        "validation_metrics": valid_metrics,
    }


def feature_name_to_source_feature(transformed_feature):
    clean_name = (
        transformed_feature
        .replace("numeric__", "")
        .replace("categorical__", "")
    )

    if clean_name.startswith("missingindicator_"):
        return clean_name.replace("missingindicator_", "")

    for column in categorical_columns:
        if clean_name == column or clean_name.startswith(f"{column}_"):
            return column

    return clean_name


def aggregate_lgbm_gain_importance(pipeline):
    transformed_feature_names = pipeline.named_steps[
        "preprocessor"
    ].get_feature_names_out()

    gain_importances = pipeline.named_steps[
        "model"
    ].feature_importances_

    importance_df = pd.DataFrame({
        "transformed_feature": transformed_feature_names,
        "gain_importance": gain_importances,
    })

    importance_df["source_feature"] = importance_df[
        "transformed_feature"
    ].map(feature_name_to_source_feature)

    source_importance_df = (
        importance_df
        .groupby("source_feature", as_index=False)
        .agg(
            gain_importance=("gain_importance", "sum"),
            transformed_feature_count=("transformed_feature", "count"),
        )
    )

    total_gain = source_importance_df["gain_importance"].sum()
    source_importance_df["gain_share"] = (
        source_importance_df["gain_importance"] / total_gain
    )

    return source_importance_df.sort_values(
        "gain_importance",
        ascending=False,
    ).reset_index(drop=True)


In [8]:
baseline_result = evaluate_feature_list(
    feature_columns=X_train.columns.tolist(),
    label="all_curated_engineered_features",
)

baseline_pipeline = baseline_result["pipeline"]
baseline_threshold = baseline_result["threshold"]
baseline_validation_metrics = baseline_result["validation_metrics"]

baseline_validation_metrics


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


{'threshold': 0.6798187129352671,
 'roc_auc': 0.759463258061794,
 'average_precision': 0.24141280853244867,
 'accuracy': 0.8532783220194301,
 'precision_class_1': 0.24848954298993028,
 'recall_class_1': 0.4038267875125881,
 'f1_class_1': 0.30766279850388417}

In [9]:
gain_importance_df = aggregate_lgbm_gain_importance(
    baseline_pipeline
)

gain_importance_df.head(20)


,source_feature,gain_importance,transformed_feature_count,gain_share
0,EXT_SOURCE_MEAN,622520.016636,2,0.407214
1,CREDIT_TERM_APPROX,144862.067506,2,0.094760
2,EXT_SOURCE_MIN,83781.842870,2,0.054805
3,EXT_SOURCE_MAX,74803.878019,2,0.048932
4,EXT_SOURCE_3,57226.031516,2,0.037434
5,GOODS_CREDIT_RATIO,55996.229835,2,0.036629
6,AGE_YEARS,55687.253870,1,0.036427
7,NAME_EDUCATION_TYPE,31059.833018,5,0.020317
8,EXT_SOURCE_STD,30733.786074,2,0.020104
9,AMT_GOODS_PRICE,30596.237345,2,0.020014


In [10]:
permutation_result = permutation_importance(
    baseline_pipeline,
    X_valid,
    y_valid,
    scoring="roc_auc",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

permutation_importance_df = pd.DataFrame({
    "source_feature": X_valid.columns,
    "permutation_importance_mean": permutation_result.importances_mean,
    "permutation_importance_std": permutation_result.importances_std,
}).sort_values(
    "permutation_importance_mean",
    ascending=False,
).reset_index(drop=True)

permutation_importance_df.head(20)


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,source_feature,permutation_importance_mean,permutation_importance_std
0,EXT_SOURCE_MEAN,0.069517,0.002444
1,CREDIT_TERM_APPROX,0.018692,0.001196
2,AGE_YEARS,0.005791,0.000807
3,GOODS_CREDIT_RATIO,0.004764,0.000484
4,AMT_GOODS_PRICE,0.004194,0.000355
5,NAME_EDUCATION_TYPE,0.004085,0.000400
6,EXT_SOURCE_3,0.004074,0.000718
7,CODE_GENDER,0.003642,0.000314
8,EXT_SOURCE_1,0.002899,0.000371
9,EXT_SOURCE_MIN,0.002276,0.000291


In [11]:
combined_importance_df = (
    gain_importance_df
    .merge(
        permutation_importance_df,
        on="source_feature",
        how="outer",
    )
    .fillna({
        "gain_importance": 0,
        "gain_share": 0,
        "transformed_feature_count": 0,
        "permutation_importance_mean": 0,
        "permutation_importance_std": 0,
    })
)

combined_importance_df["importance_rank_gain"] = combined_importance_df[
    "gain_importance"
].rank(ascending=False, method="min")

combined_importance_df["importance_rank_permutation"] = combined_importance_df[
    "permutation_importance_mean"
].rank(ascending=False, method="min")

combined_importance_df = combined_importance_df.sort_values(
    ["permutation_importance_mean", "gain_importance"],
    ascending=False,
).reset_index(drop=True)

combined_importance_df


,source_feature,gain_importance,transformed_feature_count,gain_share,permutation_importance_mean,permutation_importance_std,importance_rank_gain,importance_rank_permutation
0,EXT_SOURCE_MEAN,622520.016636,2,0.407214,0.069517,0.002444,1.0,1.0
1,CREDIT_TERM_APPROX,144862.067506,2,0.094760,0.018692,0.001196,2.0,2.0
2,AGE_YEARS,55687.253870,1,0.036427,0.005791,0.000807,7.0,3.0
3,GOODS_CREDIT_RATIO,55996.229835,2,0.036629,0.004764,0.000484,6.0,4.0
4,AMT_GOODS_PRICE,30596.237345,2,0.020014,0.004194,0.000355,10.0,5.0
5,NAME_EDUCATION_TYPE,31059.833018,5,0.020317,0.004085,0.000400,8.0,6.0
6,EXT_SOURCE_3,57226.031516,2,0.037434,0.004074,0.000718,5.0,7.0
7,CODE_GENDER,23907.556999,3,0.015639,0.003642,0.000314,17.0,8.0
8,EXT_SOURCE_1,27745.254043,2,0.018149,0.002899,0.000371,12.0,9.0
9,EXT_SOURCE_MIN,83781.842870,2,0.054805,0.002276,0.000291,3.0,10.0


## Low-Relation Feature Candidates

A feature is treated as weak only if it has very low gain contribution and non-positive or near-zero validation permutation importance. This conservative rule avoids dropping features that LightGBM uses in interaction even if permutation importance is noisy.


In [12]:
low_relation_features = combined_importance_df.loc[
    (
        combined_importance_df["permutation_importance_mean"] <= 0.00005
    )
    & (
        combined_importance_df["gain_share"] <= 0.005
    ),
    "source_feature",
].tolist()

# Keep obvious domain features unless both importance methods say they are weak.
protected_features = {
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MIN",
    "EXT_SOURCE_MAX",
    "CREDIT_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
}

low_relation_features = [
    feature for feature in low_relation_features
    if feature not in protected_features
]

low_relation_features


['NAME_HOUSING_TYPE', 'CNT_FAM_MEMBERS', 'CNT_CHILDREN', 'CHILDREN_RATIO']

In [13]:
candidate_feature_sets = {
    "all_32_features": X_train.columns.tolist(),
    "drop_low_relation_features": [
        feature for feature in X_train.columns
        if feature not in low_relation_features
    ],
    "top_24_by_permutation": permutation_importance_df.head(24)[
        "source_feature"
    ].tolist(),
    "top_20_by_permutation": permutation_importance_df.head(20)[
        "source_feature"
    ].tolist(),
}

candidate_results = []
fitted_candidates = {}

for label, feature_columns in candidate_feature_sets.items():
    print(f"Training: {label} with {len(feature_columns)} features")
    result = evaluate_feature_list(feature_columns, label)
    fitted_candidates[label] = result
    metrics = result["validation_metrics"]
    candidate_results.append({
        "feature_set": label,
        "feature_count": len(feature_columns),
        "threshold": result["threshold"],
        "validation_roc_auc": metrics["roc_auc"],
        "validation_average_precision": metrics["average_precision"],
        "validation_precision_class_1": metrics["precision_class_1"],
        "validation_recall_class_1": metrics["recall_class_1"],
        "validation_f1_class_1": metrics["f1_class_1"],
    })

candidate_results_df = pd.DataFrame(candidate_results).sort_values(
    ["validation_roc_auc", "validation_f1_class_1"],
    ascending=False,
).reset_index(drop=True)

candidate_results_df


Training: all_32_features with 32 features


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training: drop_low_relation_features with 28 features


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training: top_24_by_permutation with 24 features


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training: top_20_by_permutation with 20 features


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,feature_set,feature_count,threshold,validation_roc_auc,validation_average_precision,validation_precision_class_1,validation_recall_class_1,validation_f1_class_1
0,top_24_by_permutation,24,0.636810,0.759686,0.240142,0.225074,0.476838,0.305804
1,drop_low_relation_features,28,0.680518,0.759578,0.239712,0.247394,0.400302,0.305799
2,top_20_by_permutation,20,0.672006,0.759492,0.239641,0.241869,0.415660,0.305797
3,all_32_features,32,0.679819,0.759463,0.241413,0.248490,0.403827,0.307663


In [14]:
best_feature_set_name = candidate_results_df.loc[0, "feature_set"]
best_feature_columns = candidate_feature_sets[best_feature_set_name]
best_threshold = candidate_results_df.loc[0, "threshold"]

best_feature_set_name, len(best_feature_columns), best_threshold, best_feature_columns


('top_24_by_permutation',
 24,
 np.float64(0.6368100071556928),
 ['EXT_SOURCE_MEAN',
  'CREDIT_TERM_APPROX',
  'AGE_YEARS',
  'GOODS_CREDIT_RATIO',
  'AMT_GOODS_PRICE',
  'NAME_EDUCATION_TYPE',
  'EXT_SOURCE_3',
  'CODE_GENDER',
  'EXT_SOURCE_1',
  'EXT_SOURCE_MIN',
  'NAME_FAMILY_STATUS',
  'AMT_ANNUITY',
  'EXT_SOURCE_MAX',
  'ANNUITY_INCOME_RATIO',
  'EMPLOYMENT_AGE_RATIO',
  'EXT_SOURCE_COUNT',
  'AMT_CREDIT',
  'OCCUPATION_TYPE',
  'DAYS_EMPLOYED',
  'EXT_SOURCE_2',
  'NAME_INCOME_TYPE',
  'EMPLOYED_YEARS',
  'CREDIT_PER_PERSON',
  'ANNUITY_PER_PERSON'])

In [15]:
final_pipeline = build_lgbm_pipeline(best_feature_columns)
final_pipeline.fit(
    X_train_full[best_feature_columns],
    y_train_full,
)

test_probabilities = get_positive_probabilities(
    final_pipeline,
    X_test[best_feature_columns],
)

final_default_metrics = evaluate_probabilities(
    y_true=y_test,
    probabilities=test_probabilities,
    threshold=0.5,
)

final_selected_metrics = evaluate_probabilities(
    y_true=y_test,
    probabilities=test_probabilities,
    threshold=best_threshold,
)

final_test_df = pd.DataFrame([
    {
        "feature_set": best_feature_set_name,
        "threshold_strategy": "default_0.5",
        **final_default_metrics,
    },
    {
        "feature_set": best_feature_set_name,
        "threshold_strategy": "validation_selected",
        **final_selected_metrics,
    },
])

final_test_df


C:\Users\erenb\Desktop\Summer2026\DataScience\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,feature_set,threshold_strategy,threshold,roc_auc,average_precision,accuracy,precision_class_1,recall_class_1,f1_class_1
0,top_24_by_permutation,default_0.5,0.50000,0.764538,0.254036,0.708014,0.171213,0.681370,0.273661
1,top_24_by_permutation,validation_selected,0.63681,0.764538,0.254036,0.828968,0.232569,0.486405,0.314678


In [16]:
for threshold_strategy, threshold in [
    ("default_0.5", 0.5),
    ("validation_selected", best_threshold),
]:
    predictions = (test_probabilities >= threshold).astype(int)

    print(threshold_strategy)
    print("threshold:", threshold)
    print(confusion_matrix(y_test, predictions))
    print(classification_report(y_test, predictions, zero_division=0))


default_0.5
threshold: 0.5
[[40162 16376]
 [ 1582  3383]]
              precision    recall  f1-score   support

           0       0.96      0.71      0.82     56538
           1       0.17      0.68      0.27      4965

    accuracy                           0.71     61503
   macro avg       0.57      0.70      0.55     61503
weighted avg       0.90      0.71      0.77     61503

validation_selected
threshold: 0.6368100071556928
[[48569  7969]
 [ 2550  2415]]
              precision    recall  f1-score   support

           0       0.95      0.86      0.90     56538
           1       0.23      0.49      0.31      4965

    accuracy                           0.83     61503
   macro avg       0.59      0.67      0.61     61503
weighted avg       0.89      0.83      0.85     61503



In [17]:
final_gain_importance_df = aggregate_lgbm_gain_importance(
    final_pipeline
)

final_feature_summary = combined_importance_df.merge(
    profile_df[["feature", "p_missing", "dominant_ratio", "skewness"]],
    left_on="source_feature",
    right_on="feature",
    how="left",
).drop(columns=["feature"])

final_feature_summary["kept_in_final_model"] = final_feature_summary[
    "source_feature"
].isin(best_feature_columns)

final_feature_summary.sort_values(
    ["kept_in_final_model", "permutation_importance_mean", "gain_importance"],
    ascending=[False, False, False],
).reset_index(drop=True)


,source_feature,gain_importance,transformed_feature_count,gain_share,permutation_importance_mean,permutation_importance_std,importance_rank_gain,importance_rank_permutation,p_missing,dominant_ratio,skewness,kept_in_final_model
0,EXT_SOURCE_MEAN,622520.016636,2,0.407214,0.069517,0.002444,1.0,1.0,NaN,NaN,NaN,True
1,CREDIT_TERM_APPROX,144862.067506,2,0.094760,0.018692,0.001196,2.0,2.0,NaN,NaN,NaN,True
2,AGE_YEARS,55687.253870,1,0.036427,0.005791,0.000807,7.0,3.0,NaN,NaN,NaN,True
3,GOODS_CREDIT_RATIO,55996.229835,2,0.036629,0.004764,0.000484,6.0,4.0,NaN,NaN,NaN,True
4,AMT_GOODS_PRICE,30596.237345,2,0.020014,0.004194,0.000355,10.0,5.0,0.000904,0.084621,1.349000,True
5,NAME_EDUCATION_TYPE,31059.833018,5,0.020317,0.004085,0.000400,8.0,6.0,0.000000,0.710189,NaN,True
6,EXT_SOURCE_3,57226.031516,2,0.037434,0.004074,0.000718,5.0,7.0,0.198253,0.198253,-0.409390,True
7,CODE_GENDER,23907.556999,3,0.015639,0.003642,0.000314,17.0,8.0,0.000000,0.658344,NaN,True
8,EXT_SOURCE_1,27745.254043,2,0.018149,0.002899,0.000371,12.0,9.0,0.563811,0.563811,-0.068755,True
9,EXT_SOURCE_MIN,83781.842870,2,0.054805,0.002276,0.000291,3.0,10.0,NaN,NaN,NaN,True


## Summary

This notebook reads the profiling JSON, uses it to identify data-quality risks, uses LightGBM gain and validation permutation importance to identify weak features, tests pruned feature sets, and reports the final holdout metrics. The earlier notebooks and current LightGBM setup are unchanged.
